# DDoS / DoS Attack Detection — CICIoMT2024 (Google Colab)

## SMOTE Ratio Experiments · Feed-Forward Neural Network (FFNN)

| Item | Detail |
|------|--------|
| **Platform** | Google Colab — **Runtime → Run all** |
| **GPU** | **Runtime → Change runtime type → T4 GPU** (recommended for FFNN + XGBoost) |
| **Dataset** | Auto-download from Kaggle via Colab Secrets |
| **Kaggle auth** | Secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY` |
| **Pipeline** | Preprocess → Train/Test split → **Resample train only** → Train FFNN → Evaluate test |
| **SMOTE ratios** | 10:1 · 5:1 · 2:1 · 1:1 (Class 0 : Class 1) |
| **Arch sweep** | 6 hidden-layer configs at best SMOTE ratio |
| **Output** | `/content/outputs/` and `/content/models/` |

> **Binary label:** `1` if label contains `ddos` or `dos`, else `0`.
> Test set is **never** resampled. FFNN module is inlined — no `src/ffnn.py` upload needed.


---
## Cell 1 — GPU Check & Install Packages


In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU detected (FFNN will use PyTorch CUDA when available):')
    for line in result.stdout.split('\n'):
        if any(k in line for k in ('NVIDIA', 'Tesla', 'GeForce')):
            print(' ', line.strip())
else:
    print('No GPU — enable Runtime > Change runtime type > T4 GPU.')

!pip install -q --upgrade torch xgboost scikit-learn imbalanced-learn psutil joblib kaggle kagglehub

import torch
import sklearn
print(f'\nPyTorch : {torch.__version__} (CUDA: {torch.cuda.is_available()})')
print(f'Sklearn : {sklearn.__version__}')
print('Packages ready.')


---
## Cell 2 — Kaggle Auth & Dataset Download

1. Open Colab **Secrets** (sidebar key icon).
2. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` from [kaggle.com/settings](https://www.kaggle.com/settings).
3. Enable **Notebook access** for both secrets.
4. Re-run this cell if download fails after adding secrets.


In [ ]:
import json, os, subprocess
from pathlib import Path

KAGGLE_DATASET = 'limamateus/cic-iomt-2024-wifi-mqtt'
DATA_DIR  = Path('/content/ciciomt2024')
MODEL_DIR = Path('/content/models')
OUTPUT_DIR = Path('/content/outputs')
for p in (DATA_DIR, MODEL_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_train.csv'
TEST_FILE  = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_test.csv'


def setup_kaggle_credentials():
    kaggle_dir = Path('/root/.kaggle')
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists():
        print('Using existing /root/.kaggle/kaggle.json')
        return

    username, key = None, None
    try:
        from google.colab import userdata
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
        print('Kaggle credentials loaded from Colab Secrets.')
    except Exception:
        username = os.environ.get('KAGGLE_USERNAME')
        key = os.environ.get('KAGGLE_KEY')
        if username and key:
            print('Kaggle credentials loaded from environment variables.')

    if not username or not key:
        raise RuntimeError(
            'Kaggle credentials not found.\n'
            'Colab sidebar > Secrets > add KAGGLE_USERNAME and KAGGLE_KEY, '
            'then enable Notebook access.'
        )

    kaggle_dir.mkdir(parents=True, exist_ok=True)
    with open(kaggle_json, 'w', encoding='utf-8') as f:
        json.dump({'username': username, 'key': key}, f)
    os.chmod(kaggle_json, 0o600)
    print('kaggle.json created.')


def ensure_dataset(require_test=True):
    global TRAIN_FILE, TEST_FILE
    if TRAIN_FILE.exists() and (not require_test or TEST_FILE.exists()):
        print(f'Dataset already present in {DATA_DIR}')
        return

    setup_kaggle_credentials()
    print(f'Downloading {KAGGLE_DATASET} ...')
    try:
        import kagglehub
        dl_path = kagglehub.dataset_download(KAGGLE_DATASET)
        print(f'kagglehub path: {dl_path}')
        for csv in Path(dl_path).rglob('*.csv'):
            target = DATA_DIR / csv.name
            if not target.exists():
                target.write_bytes(csv.read_bytes())
    except Exception as e1:
        print(f'kagglehub failed ({e1}), trying kaggle CLI...')
        subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET,
             '-p', str(DATA_DIR), '--unzip', '-q'],
            check=False,
        )

    if not TRAIN_FILE.exists():
        found = list(DATA_DIR.rglob('*train*.csv'))
        if found:
            TRAIN_FILE = found[0]
        else:
            raise FileNotFoundError(f'Train CSV not found under {DATA_DIR}')
    if require_test and not TEST_FILE.exists():
        found = list(DATA_DIR.rglob('*test*.csv'))
        if found:
            TEST_FILE = found[0]
        else:
            raise FileNotFoundError(f'Test CSV not found under {DATA_DIR}')

    print(f'\nTrain: {TRAIN_FILE}')
    if require_test:
        print(f'Test : {TEST_FILE}')


ensure_dataset(require_test=True)


---
## Cell 3 — Imports, FFNN Module & Settings

Paths are set in Cell 2 (`/content/ciciomt2024`, `/content/models`, `/content/outputs`).
The FFNN module is inlined below (no external `src/ffnn.py` required on Colab).


In [ ]:
# ── Step 1: Standard library & data-science imports ─────────────────────────
import os, sys, json, time, warnings, tracemalloc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_fscore_support,
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import torch
from xgboost import XGBClassifier


# ── FFNN module (inlined for Colab — mirrors src/ffnn.py) ───────────────────
import copy
from typing import Any

import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

FFNN_PARAMS = {
    'hidden_layers': (128, 64, 32),
    'dropout': 0.3,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'batch_size': 4096,
    'max_epochs': 100,
    'early_stopping_patience': 25,
    'random_state': 42,
    'verbose': False,
}


def detect_ffnn_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


class FeedForwardNet(nn.Module):
    def __init__(self, n_features, hidden_layers=(128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        in_dim = n_features
        for hidden_dim in hidden_layers:
            layers.extend([nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)


class FFNNClassifier:
    def __init__(
        self,
        hidden_layers=(128, 64, 32),
        dropout=0.3,
        learning_rate=1e-3,
        weight_decay=1e-4,
        batch_size=4096,
        max_epochs=100,
        early_stopping_patience=25,
        random_state=42,
        device=None,
        verbose=False,
    ):
        self.hidden_layers = hidden_layers
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.early_stopping_patience = early_stopping_patience
        self.random_state = random_state
        self.device = torch.device(device) if device is not None else detect_ffnn_device()
        self.verbose = verbose
        self.model_ = None
        self.n_features_ = None
        self.best_epoch_ = 0
        self.best_val_f1_ = 0.0

    def _to_numpy(self, X):
        if isinstance(X, pd.DataFrame):
            return X.to_numpy(dtype=np.float32)
        return np.asarray(X, dtype=np.float32)

    def _to_tensor(self, X):
        return torch.from_numpy(self._to_numpy(X)).to(self.device)

    def _make_loader(self, X, y, shuffle):
        x_tensor = self._to_tensor(X)
        y_tensor = torch.from_numpy(np.asarray(y, dtype=np.float32)).to(self.device)
        return DataLoader(TensorDataset(x_tensor, y_tensor), batch_size=self.batch_size, shuffle=shuffle)

    @torch.no_grad()
    def _predict_proba_array(self, X):
        if self.model_ is None:
            raise RuntimeError('Model is not fitted.')
        self.model_.eval()
        loader = self._make_loader(X, np.zeros(len(X)), shuffle=False)
        probs = []
        for x_batch, _ in loader:
            logits = self.model_(x_batch)
            probs.append(torch.sigmoid(logits).cpu().numpy())
        pos_prob = np.concatenate(probs)
        return np.column_stack([1.0 - pos_prob, pos_prob])

    def _evaluate_f1(self, X_val, y_val):
        proba = self._predict_proba_array(X_val)[:, 1]
        preds = (proba >= 0.5).astype(int)
        return float(f1_score(y_val, preds, zero_division=0))

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        x_train, x_val, y_train, y_val = train_test_split(
            X, y, test_size=0.1, random_state=self.random_state, stratify=y,
        )
        self.n_features_ = x_train.shape[1]
        self.model_ = FeedForwardNet(
            n_features=self.n_features_,
            hidden_layers=self.hidden_layers,
            dropout=self.dropout,
        ).to(self.device)
        pos = max(int(np.sum(y_train == 1)), 1)
        neg = max(int(np.sum(y_train == 0)), 1)
        pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=self.device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(
            self.model_.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay,
        )
        train_loader = self._make_loader(x_train, y_train, shuffle=True)
        best_state = copy.deepcopy(self.model_.state_dict())
        best_f1, best_epoch, rounds_no_improve = -1.0, 0, 0
        for epoch in range(1, self.max_epochs + 1):
            self.model_.train()
            for x_batch, y_batch in train_loader:
                optimizer.zero_grad()
                loss = criterion(self.model_(x_batch), y_batch)
                loss.backward()
                optimizer.step()
            val_f1 = self._evaluate_f1(x_val, y_val)
            if val_f1 > best_f1:
                best_f1, best_epoch = val_f1, epoch
                best_state = copy.deepcopy(self.model_.state_dict())
                rounds_no_improve = 0
            else:
                rounds_no_improve += 1
                if rounds_no_improve >= self.early_stopping_patience:
                    break
        self.model_.load_state_dict(best_state)
        self.best_epoch_, self.best_val_f1_ = best_epoch, best_f1
        if self.verbose:
            label = 'GPU' if self.device.type == 'cuda' else 'CPU'
            print(f'    early stop @ epoch {best_epoch} (val F1={best_f1:.4f}, {label})')
        return self

    def predict_proba(self, X):
        return self._predict_proba_array(X)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def create_ffnn(**overrides):
    params = {**FFNN_PARAMS, **overrides}
    return FFNNClassifier(**params)


def fit_ffnn(model, X, y):
    model.verbose = True
    model.fit(X, y)


def per_class_recall(y_true, y_pred):
    rec = recall_score(y_true, y_pred, labels=[0, 1], average=None, zero_division=0)
    return round(float(rec[0]), 4), round(float(rec[1]), 4)


def build_smote_experiment_row(
    *, model_name, y_test, y_pred, y_prob, ratio_label, resample_method,
    train_rows, train_time, n_features,
):
    rec0, rec1 = per_class_recall(y_test, y_pred)
    return {
        'Model': model_name,
        'SMOTE_Ratio': ratio_label,
        'Resample_Method': resample_method,
        'Train_Rows_After_Resample': train_rows,
        'Class_0_Recall': rec0,
        'Class_1_Recall': rec1,
        'Accuracy': round(float(accuracy_score(y_test, y_pred)), 4),
        'F1-Score': round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
        'AUC': round(float(roc_auc_score(y_test, y_prob)), 4),
        'Train_Time_s': round(train_time, 2),
        'n_features': n_features,
    }


def print_smote_experiment_row(row):
    print(f"  Class 0 Recall (Other)    : {row['Class_0_Recall']}")
    print(f"  Class 1 Recall (DDoS/DoS) : {row['Class_1_Recall']}")
    print(f"  Accuracy                  : {row['Accuracy']}")
    print(f"  F1-Score                  : {row['F1-Score']}")
    print(f"  AUC                       : {row['AUC']}")
    print(f"  Train time                : {row['Train_Time_s']}s")


try:
    from IPython.display import display
except ImportError:
    display = print

# ── Step 2: Colab paths (also set in Kaggle download cell) ─
DATA_DIR  = Path('/content/ciciomt2024')
MODEL_DIR = Path('/content/models')
OUTPUT_DIR = Path('/content/outputs')
for p in (DATA_DIR, MODEL_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Colab paths set in Cell 2 (Kaggle download)

TRAIN_FILE = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_train.csv'
TEST_FILE  = DATA_DIR / 'CIC_IoMT_2024_WiFi_MQTT_test.csv'

RANDOM_SEED = 42
MAX_ROWS = int(os.getenv('MAX_ROWS', '500000'))
TOP_N = 20
SMOTE_RATIOS = ["10:1", "5:1", "2:1", "1:1"]  # Class 0 : Class 1
EARLY_STOPPING_ROUNDS = 25   # shared patience — matches XGBoost
SKLEARN_MAX_ESTIMATORS = 300   # upper cap for RF / AdaBoost warm_start search
CV_FOLDS = int(os.getenv('CV_FOLDS', '3'))
CV_MAX_ROWS = int(os.getenv('CV_MAX_ROWS', '150000'))
np.random.seed(RANDOM_SEED)

try:
    XGBClassifier(device='cuda', n_estimators=1)
    XGB_DEVICE = 'cuda'
    print('XGBoost feature selector will use GPU (device=cuda)')
except Exception:
    XGB_DEVICE = 'cpu'
    print('XGBoost feature selector will use CPU')

print(f'Data dir   : {DATA_DIR}')
print(f'Model dir   : {MODEL_DIR}')
print(f'Output dir  : {OUTPUT_DIR}')

FFNN_DEVICE = detect_ffnn_device()
print(f'FFNN device : {FFNN_DEVICE}')


---
## Cell 4 — Load Dataset

Train/test CSVs are downloaded in Cell 2. This cell loads them into memory.


In [ ]:
def load_with_label(filepath):
    """Load a CSV and normalize its label column to 'label'."""
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()
    existing = [c for c in df.columns if c.lower() in ('label', 'class', 'attack', 'type')]
    if existing:
        df.rename(columns={existing[0]: 'label'}, inplace=True)
    else:
        stem = Path(filepath).stem
        for sfx in ('_train', '_test', '-train', '-test'):
            stem = stem.replace(sfx, '')
        df['label'] = stem
    df['label'] = df['label'].astype(str).str.strip()
    return df

print(f'\nTrain: {TRAIN_FILE}')
print(f'Test : {TEST_FILE}')

print('Loading TRAIN...')
df_train = load_with_label(TRAIN_FILE)
print(f'  Train: {len(df_train):,} rows x {df_train.shape[1]} cols')

df_test = None
if TEST_FILE.exists():
    print('Loading TEST...')
    df_test = load_with_label(TEST_FILE)
    common = sorted(set(df_train.columns) & set(df_test.columns))
    df_train = df_train[common]
    df_test = df_test[common]
    print(f'  Test : {len(df_test):,} rows x {df_test.shape[1]} cols')
else:
    print('No official test file — an 80/20 split will be used later.')


---
## Cell 4 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Step 1: Basic dataset health checks ─────────────────────────────────────
print('=== DATASET OVERVIEW ===')
print(f'Rows     : {len(df_train):,}')
print(f'Columns  : {df_train.shape[1]}')
print(f'Memory   : {df_train.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'Missing  : {df_train.isnull().sum().sum():,}')
print(f'Duplicates: {df_train.duplicated().sum():,}')
print('\nDtypes:')
print(df_train.dtypes.value_counts())

# ── Step 2: Show how many rows belong to each raw attack label ───────────────
print('\n=== TOP 20 RAW LABELS ===')
label_counts = df_train['label'].value_counts()
for lbl, cnt in label_counts.head(20).items():
    print(f'  {lbl:<45} {cnt:>10,}  ({100*cnt/len(df_train):5.2f}%)')

# ── Step 3: Visualize label distribution and feature scatter sample ───────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
label_counts.head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Attack Labels (Raw)')
axes[0].set_xlabel('Count')

numeric_cols = df_train.select_dtypes(include=[np.number]).columns[:6]
if len(numeric_cols) >= 2:
    sample = df_train.sample(min(5000, len(df_train)), random_state=RANDOM_SEED)
    sns.scatterplot(data=sample, x=numeric_cols[0], y=numeric_cols[1],
                    hue='label', legend=False, alpha=0.4, ax=axes[1])
    axes[1].set_title(f'{numeric_cols[0]} vs {numeric_cols[1]} (sample)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 5 — Binary Label: DDoS or DoS vs Other

In [ ]:
# ── Step 1: Map multi-class labels to binary attack vs non-attack ────────────
def to_attack_binary(label: str) -> int:
    """Return 1 if label mentions ddos/dos; otherwise 0 (benign/other attacks)."""
    text = str(label).lower()
    return 1 if ('ddos' in text or 'dos' in text) else 0

df_train['binary_label'] = df_train['label'].apply(to_attack_binary)
if df_test is not None:
    df_test['binary_label'] = df_test['label'].apply(to_attack_binary)

# ── Step 2: Report class balance on training data ───────────────────────────
counts = df_train['binary_label'].value_counts().sort_index()
print('Binary distribution (train):')
print(f'  Other (0) : {counts.get(0, 0):,}')
print(f'  DDoS/DoS(1): {counts.get(1, 0):,}')
print(f'  Imbalance : {counts.max()/max(counts.min(),1):.1f}x')

# ── Step 3: Plot binary distribution for the report ───────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_xticklabels(['Other (0)', 'DDoS/DoS (1)'], rotation=0)
ax.set_title('Binary Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'binary_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 6 — Preprocessing Pipeline (no resampling)


In [ ]:
# Columns that are identifiers/labels — never used as model features
DROP_COLS = ['label', 'binary_label', 'flow_id', 'Flow ID',
             'src_ip', 'Src IP', 'dst_ip', 'Dst IP']

def get_Xy(df, target='binary_label'):
    """Split dataframe into feature matrix X and target vector y."""
    drop = [c for c in DROP_COLS if c in df.columns]
    X = df.drop(columns=drop).copy()
    y = df[target].copy().reset_index(drop=True)
    return X, y

# ── Step 1: Extract features/target from training dataframe ─────────────────
X_raw, y = get_Xy(df_train)
y_groups = df_train['label'].astype(str).str.strip().reset_index(drop=True)

# ── Step 2: Label-encode categoricals (fit encoders on train only) ──────────
label_encoders = {}
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    X_raw[col] = le.fit_transform(X_raw[col].astype(str))
    label_encoders[col] = le

# ── Step 3: Clean infinities/NaNs; drop mostly-empty columns; impute medians
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
miss_pct = X_raw.isnull().mean()
drop_miss = miss_pct[miss_pct > 0.5].index.tolist()
if drop_miss:
    X_raw.drop(columns=drop_miss, inplace=True)
train_medians = X_raw.median(numeric_only=True)
X_raw.fillna(train_medians, inplace=True)
X_raw = X_raw.select_dtypes(include=[np.number]).reset_index(drop=True)
feature_columns = X_raw.columns.tolist()
print(f'Numeric features: {len(feature_columns)}')

# ── Step 4: Remove exact duplicate feature rows ─────────────────────────────
mask = ~X_raw.duplicated()
X_raw, y = X_raw[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
y_groups = y_groups[mask].reset_index(drop=True)
print(f'After dedup: {len(X_raw):,} rows')

# ── Step 5: Stratified down-sample if dataset exceeds MAX_ROWS ───────────────
if len(X_raw) > MAX_ROWS:
    print(f'Sampling {MAX_ROWS:,} from {len(X_raw):,} rows...')
    rng = np.random.default_rng(RANDOM_SEED)
    parts = []
    for cls in y.unique():
        idx = y[y == cls].index.to_numpy()
        n_take = int(MAX_ROWS * len(idx) / len(y))
        n_take = min(n_take, len(idx))
        parts.append(rng.choice(idx, n_take, replace=False))
    sampled_idx = np.concatenate(parts)
    rng.shuffle(sampled_idx)
    X_raw = X_raw.iloc[sampled_idx].reset_index(drop=True)
    y = y.iloc[sampled_idx].reset_index(drop=True)
    y_groups = y_groups.iloc[sampled_idx].reset_index(drop=True)
print(f'Working set: {len(X_raw):,} rows')

# ── Step 6: Standardize features (fit scaler on train only) ─────────────────
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)

# ── Step 7: Build train/test matrices (official split or 80/20 stratified) ───
if df_test is not None:
    X_test_raw, y_test = get_Xy(df_test)
    for col, le in label_encoders.items():
        if col in X_test_raw.columns:
            X_test_raw[col] = (
                X_test_raw[col].astype(str)
                .apply(lambda v: int(le.transform([v])[0]) if v in le.classes_ else 0)
            )
    X_test_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_test_raw.fillna(train_medians, inplace=True)
    X_test_raw = X_test_raw[[c for c in feature_columns if c in X_test_raw.columns]]
    X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=X_test_raw.columns)
    X_train, y_train = X_scaled, y
    label_groups_train = y_groups.reset_index(drop=True)
    print('Using official CIC train/test split.')
else:
    train_idx, test_idx = train_test_split(
        np.arange(len(X_scaled)), test_size=0.2, random_state=RANDOM_SEED, stratify=y)
    X_train = X_scaled.iloc[train_idx].reset_index(drop=True)
    X_test  = X_scaled.iloc[test_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_test  = y.iloc[test_idx].reset_index(drop=True)
    label_groups_train = y_groups.iloc[train_idx].reset_index(drop=True)
    print('Using 80/20 stratified split.')


# ── Step 8: Leakage guard — drop train rows identical to official test flows ──
if df_test is not None:
    common_feat = [c for c in feature_columns if c in X_test_raw.columns]
    train_sig = pd.util.hash_pandas_object(
        X_raw[common_feat].fillna(-999).astype(str), index=False)
    test_sig = pd.util.hash_pandas_object(
        X_test_raw[common_feat].fillna(-999).astype(str), index=False)
    overlap_mask = train_sig.isin(set(test_sig))
    n_overlap = int(overlap_mask.sum())
    if n_overlap:
        print(f'Removing {n_overlap:,} train rows that duplicate official test flows (leakage guard).')
        keep = ~overlap_mask.to_numpy()
        X_raw = X_raw.loc[keep].reset_index(drop=True)
        y = y.loc[keep].reset_index(drop=True)
        y_groups = y_groups.loc[keep].reset_index(drop=True)
        X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)
        X_train, y_train = X_scaled, y
        label_groups_train = y_groups.reset_index(drop=True)
    else:
        print('Train/test feature-vector overlap: 0 rows (no cross-split duplicates).')

# ── Step 9: Report class balance (NO resampling here — done per experiment) ───
neg_train = int((y_train == 0).sum())
pos_train = int((y_train == 1).sum())
XGB_SCALE_POS_WEIGHT = neg_train / max(pos_train, 1)
majority_baseline = max(y_test.value_counts()) / len(y_test)
train_ratio = neg_train / max(pos_train, 1)
print(f'Train class counts — Other(0): {neg_train:,} | DDoS/DoS(1): {pos_train:,} | ratio {train_ratio:.2f}:1')
print(f'XGB scale_pos_weight (pre-resample train): {XGB_SCALE_POS_WEIGHT:.4f}')
print(f'Majority-class baseline accuracy (test): {majority_baseline:.4f}')
print(f'\nTrain: {X_train.shape} | Test (untouched): {X_test.shape}')
print('Resampling (SMOTE / undersampling) is applied inside each experiment on TRAIN only.')


In [ ]:
# ── Resampling helpers (train only, after train/test split) ─────────────────

def _parse_ratio(ratio_label: str):
    """Parse '10:1' as Class0:Class1 target counts ratio."""
    parts = ratio_label.strip().split(':')
    if len(parts) != 2:
        raise ValueError(f'Expected Class0:Class1 format, got {ratio_label!r}')
    return int(parts[0]), int(parts[1])


def resample_train_to_ratio(X_train, y_train, ratio_label, random_state=RANDOM_SEED):
    """
    Resample training data to target Class0:Class1 ratio using SMOTE or undersampling.

    Pipeline: split first → resample train only → model fit → evaluate on untouched test.
    Uses dict sampling_strategy (absolute counts) — imblearn floats only allow (0, 1].
    """
    r0, r1 = _parse_ratio(ratio_label)
    target_ratio = r0 / r1  # n0 / n1

    n0 = int((y_train == 0).sum())
    n1 = int((y_train == 1).sum())
    current_ratio = n0 / max(n1, 1)

    if abs(current_ratio - target_ratio) / max(target_ratio, 1e-9) < 1e-6:
        return X_train.copy(), y_train.copy(), 'none'

    # Target counts for Class0:Class1 = r0:r1
    n0_if_keep_n1 = max(1, int(round(n1 * target_ratio)))
    n1_if_keep_n0 = max(1, int(round(n0 * r1 / r0)))

    if current_ratio > target_ratio:
        # Too much class 0 — undersample class 0 or SMOTE class 1
        if n0 > n0_if_keep_n1:
            strategy = {0: n0_if_keep_n1}
            sampler = RandomUnderSampler(sampling_strategy=strategy, random_state=random_state)
            method = 'undersample'
        else:
            strategy = {1: n1_if_keep_n0}
            k = min(5, n1 - 1)
            if k < 1:
                print(f'  Warning: too few class-1 samples for {ratio_label}; skipping resample.')
                return X_train.copy(), y_train.copy(), 'skipped'
            sampler = SMOTE(sampling_strategy=strategy, random_state=random_state, k_neighbors=k)
            method = 'SMOTE'
    else:
        # Too little class 0 — SMOTE class 0 or undersample class 1
        if n0 < n0_if_keep_n1:
            strategy = {0: n0_if_keep_n1}
            k = min(5, n0 - 1)
            if k < 1:
                print(f'  Warning: too few class-0 samples for {ratio_label}; skipping resample.')
                return X_train.copy(), y_train.copy(), 'skipped'
            sampler = SMOTE(sampling_strategy=strategy, random_state=random_state, k_neighbors=k)
            method = 'SMOTE'
        else:
            strategy = {1: n1_if_keep_n0}
            sampler = RandomUnderSampler(sampling_strategy=strategy, random_state=random_state)
            method = 'undersample'

    X_res, y_res = sampler.fit_resample(X_train, y_train)
    X_res = pd.DataFrame(X_res, columns=X_train.columns)
    y_res = pd.Series(y_res, name='binary_label').reset_index(drop=True)

    n0r = int((y_res == 0).sum())
    n1r = int((y_res == 1).sum())
    print(f'  {method} -> Other(0)={n0r:,} DDoS/DoS(1)={n1r:,} ({n0r/max(n1r,1):.2f}:1)')
    return X_res, y_res, method


def select_features_xgb(X_fit, y_fit, X_eval, scale_pos_weight):
    """XGB gain-importance top-N on resampled train; apply to train and test."""
    xgb_sel = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.2,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        device=XGB_DEVICE, random_state=RANDOM_SEED, verbosity=0,
        eval_metric='logloss')
    xgb_sel.fit(X_fit, y_fit)
    features = (
        pd.DataFrame({'feature': X_fit.columns, 'importance': xgb_sel.feature_importances_})
        .sort_values('importance', ascending=False)
        .head(TOP_N)['feature'].tolist()
    )
    return features, xgb_sel


def per_class_recall(y_true, y_pred):
    """Return recall for Class 0 (Other) and Class 1 (DDoS/DoS)."""
    rec = recall_score(y_true, y_pred, labels=[0, 1], average=None, zero_division=0)
    return round(float(rec[0]), 4), round(float(rec[1]), 4)


def fit_xgb(m, X, y):
    """Train XGBoost with a small internal validation set for early stopping."""
    Xf, Xv, yf, yv = train_test_split(
        X, y, test_size=0.1, random_state=RANDOM_SEED, stratify=y)
    m.fit(Xf, yf, eval_set=[(Xv, yv)], verbose=False)


def _train_val_split(X, y):
    """90/10 stratified split — same protocol as fit_xgb."""
    return train_test_split(
        X, y, test_size=0.1, random_state=RANDOM_SEED, stratify=y)


def fit_decision_tree(m, X, y):
    """Regularized DT — pick max_depth by validation F1 (early stopping on tree depth)."""
    Xf, Xv, yf, yv = _train_val_split(X, y)
    base_params = m.get_params()
    cap_depth = base_params.get('max_depth') or 8
    best_f1, best_depth = -1.0, 2
    rounds_no_improve = 0
    for depth in range(2, int(cap_depth) + 1):
        trial = DecisionTreeClassifier(**{**base_params, 'max_depth': depth})
        trial.fit(Xf, yf)
        f1 = f1_score(yv, trial.predict(Xv), zero_division=0)
        if f1 > best_f1:
            best_f1, best_depth = f1, depth
            rounds_no_improve = 0
        else:
            rounds_no_improve += 1
            if rounds_no_improve >= EARLY_STOPPING_ROUNDS:
                break
    m.set_params(max_depth=best_depth)
    m.fit(Xf, yf)
    print(f'    max_depth={best_depth} (val F1={best_f1:.4f})')


def fit_rf_early_stop(m, X, y, step=5, patience=None):
    """Random Forest: warm_start + validation F1 early stopping."""
    patience = patience or EARLY_STOPPING_ROUNDS
    Xf, Xv, yf, yv = _train_val_split(X, y)
    probe = clone(m)
    probe.set_params(warm_start=True, n_estimators=0)
    max_n = SKLEARN_MAX_ESTIMATORS
    best_f1, best_n = -1.0, step
    rounds_no_improve = 0
    for n in range(step, max_n + 1, step):
        probe.set_params(n_estimators=n)
        probe.fit(Xf, yf)
        f1 = f1_score(yv, probe.predict(Xv), zero_division=0)
        if f1 > best_f1:
            best_f1, best_n = f1, n
            rounds_no_improve = 0
        else:
            rounds_no_improve += 1
            if rounds_no_improve >= patience:
                break
    m.set_params(n_estimators=best_n, warm_start=False)
    m.fit(Xf, yf)
    print(f'    early stop @ {best_n} estimators (val F1={best_f1:.4f})')


def fit_adaboost_early_stop(m, X, y, patience=None):
    """AdaBoost: staged_predict validation F1 early stopping (no warm_start in sklearn 1.9)."""
    patience = patience or EARLY_STOPPING_ROUNDS
    Xf, Xv, yf, yv = _train_val_split(X, y)
    probe = clone(m)
    probe.set_params(n_estimators=SKLEARN_MAX_ESTIMATORS)
    probe.fit(Xf, yf)
    best_f1, best_n = -1.0, 1
    rounds_no_improve = 0
    for n, y_pred in enumerate(probe.staged_predict(Xv), start=1):
        f1 = f1_score(yv, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_n = f1, n
            rounds_no_improve = 0
        else:
            rounds_no_improve += 1
            if rounds_no_improve >= patience:
                break
    m.set_params(n_estimators=best_n)
    m.fit(Xf, yf)
    print(f'    early stop @ {best_n} estimators (val F1={best_f1:.4f})')


---
## Cell — FFNN SMOTE Ratio Experiments


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SMOTE ratio experiments — Feed-Forward Neural Network
# ═══════════════════════════════════════════════════════════════════════════

MODEL_NAME = 'Feed-Forward NN'
MODEL_COLOR = '#9b59b6'

SMOTE_EXPERIMENT_RESULTS = []

for ratio_label in SMOTE_RATIOS:
    print(f'\n{"="*70}\n  SMOTE ratio experiment: {ratio_label} (Class 0 : Class 1)\n{"="*70}')

    X_res, y_res, resample_method = resample_train_to_ratio(X_train, y_train, ratio_label)

    spw = 1.0  # SMOTE already set train ratio
    selected_features, _ = select_features_xgb(X_res, y_res, X_test, spw)
    X_tr = X_res[selected_features]
    X_te = X_test[selected_features]

    print(f'\n  --- {MODEL_NAME} ---')
    model = create_ffnn(device=FFNN_DEVICE)
    t0 = time.time()
    fit_ffnn(model, X_tr, y_res)
    train_time = time.time() - t0

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    row = build_smote_experiment_row(
        model_name=MODEL_NAME,
        y_test=y_test,
        y_pred=y_pred,
        y_prob=y_prob,
        ratio_label=ratio_label,
        resample_method=resample_method,
        train_rows=len(X_res),
        train_time=train_time,
        n_features=len(selected_features),
    )
    SMOTE_EXPERIMENT_RESULTS.append(row)
    print_smote_experiment_row(row)

smote_results_df = pd.DataFrame(SMOTE_EXPERIMENT_RESULTS)
print('\n=== SMOTE RATIO EXPERIMENT SUMMARY (FFNN) ===')
display(smote_results_df)
out_csv = OUTPUT_DIR / 'smote_ratio_experiments_ffnn.csv'
smote_results_df.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')


---
## Cell — FFNN Results Visualization


In [ ]:
# ── Visualize FFNN SMOTE ratio experiment results ───────────────────────────
ratio_order = SMOTE_RATIOS
metrics = ['Class_1_Recall', 'Class_0_Recall', 'F1-Score', 'AUC']

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.ravel()

for ax, metric in zip(axes, metrics):
    sub = smote_results_df.set_index('SMOTE_Ratio').reindex(ratio_order)
    sub[metric].plot(kind='bar', ax=ax, color=MODEL_COLOR, rot=0)
    ax.set_title(f'FFNN — {metric} by SMOTE Ratio')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('SMOTE Ratio (Class 0 : Class 1)')

plt.tight_layout()
plot_path = OUTPUT_DIR / 'smote_ratio_experiments_ffnn.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

# Best ratio by F1
best = smote_results_df.loc[smote_results_df['F1-Score'].idxmax()]
print('\n=== BEST FFNN CONFIG (by F1-Score) ===')
for col in ['SMOTE_Ratio', 'Class_0_Recall', 'Class_1_Recall', 'Accuracy', 'F1-Score', 'AUC', 'Train_Time_s']:
    print(f'  {col:<22}: {best[col]}')


---
## Cell — FFNN Architecture Sweep

Compare layer depth and width using the **best SMOTE ratio** from the previous section (same 20 XGB-selected features, same train/test split).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FFNN architecture sweep — vary hidden_layers at fixed SMOTE ratio
# ═══════════════════════════════════════════════════════════════════════════

def _count_ffnn_params(n_features: int, hidden_layers: tuple[int, ...]) -> int:
    """Estimate trainable parameters for FeedForwardNet (Linear + bias only)."""
    total = 0
    in_dim = n_features
    for h in hidden_layers:
        total += in_dim * h + h
        in_dim = h
    total += in_dim * 1 + 1
    return total


FFNN_ARCHITECTURES = [
    {'label': 'tiny (32)', 'hidden_layers': (32,)},
    {'label': 'small (64,32)', 'hidden_layers': (64, 32)},
    {'label': 'default (128,64,32)', 'hidden_layers': (128, 64, 32)},
    {'label': 'wide (256,128,64)', 'hidden_layers': (256, 128, 64)},
    {'label': 'deep (128,128,64,32)', 'hidden_layers': (128, 128, 64, 32)},
    {'label': 'dense (128,128,128)', 'hidden_layers': (128, 128, 128)},
]

ARCH_SWEEP_SMOTE_RATIO = smote_results_df.loc[smote_results_df['F1-Score'].idxmax(), 'SMOTE_Ratio']
print(f'Architecture sweep SMOTE ratio: {ARCH_SWEEP_SMOTE_RATIO}')

X_arch, y_arch, _ = resample_train_to_ratio(X_train, y_train, ARCH_SWEEP_SMOTE_RATIO)
arch_features, _ = select_features_xgb(X_arch, y_arch, X_test, 1.0)
X_arch_tr = X_arch[arch_features]
X_arch_te = X_test[arch_features]
n_in = len(arch_features)

ARCH_SWEEP_RESULTS = []

for spec in FFNN_ARCHITECTURES:
    label = spec['label']
    layers = spec['hidden_layers']
    print(f'\n{"="*70}\n  Architecture: {label}  layers={layers}\n{"="*70}')

    model = create_ffnn(device=FFNN_DEVICE, hidden_layers=layers)
    t0 = time.time()
    fit_ffnn(model, X_arch_tr, y_arch)
    train_time = time.time() - t0

    y_pred = model.predict(X_arch_te)
    y_prob = model.predict_proba(X_arch_te)[:, 1]
    rec0, rec1 = per_class_recall(y_test, y_pred)

    row = {
        'Architecture': label,
        'hidden_layers': str(layers),
        'n_params': _count_ffnn_params(n_in, layers),
        'SMOTE_Ratio': ARCH_SWEEP_SMOTE_RATIO,
        'Class_0_Recall': rec0,
        'Class_1_Recall': rec1,
        'Accuracy': round(float(accuracy_score(y_test, y_pred)), 4),
        'F1-Score': round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
        'AUC': round(float(roc_auc_score(y_test, y_prob)), 4),
        'Train_Time_s': round(train_time, 2),
        'best_epoch': model.best_epoch_,
        'best_val_f1': round(model.best_val_f1_, 4),
    }
    ARCH_SWEEP_RESULTS.append(row)
    print(f'  Params ~{row["n_params"]:,} | val F1={row["best_val_f1"]} @ epoch {row["best_epoch"]}')
    print(f'  Test — Acc={row["Accuracy"]} F1={row["F1-Score"]} AUC={row["AUC"]} | {row["Train_Time_s"]}s')

arch_sweep_df = pd.DataFrame(ARCH_SWEEP_RESULTS)
arch_order = [s['label'] for s in FFNN_ARCHITECTURES]

print('\n=== FFNN ARCHITECTURE SWEEP SUMMARY ===')
display(arch_sweep_df)

arch_csv = OUTPUT_DIR / 'ffnn_architecture_sweep.csv'
arch_sweep_df.to_csv(arch_csv, index=False)
print(f'Saved: {arch_csv}')

# ── Comparative visualization ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
plot_metrics = ['F1-Score', 'AUC', 'Accuracy', 'Train_Time_s']
colors = sns.color_palette('viridis', len(arch_order))

for ax, metric in zip(axes, plot_metrics):
    sub = arch_sweep_df.set_index('Architecture').reindex(arch_order)
    bars = sub[metric].plot(kind='bar', ax=ax, color=colors, rot=45)
    ax.set_title(f'FFNN Architecture — {metric}')
    if metric != 'Train_Time_s':
        ax.set_ylim(0.95, 1.005)
    ax.set_xlabel('')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f' if metric != 'Train_Time_s' else '%.0f', fontsize=7, rotation=90, padding=2)

plt.tight_layout()
arch_plot = OUTPUT_DIR / 'ffnn_architecture_sweep.png'
plt.savefig(arch_plot, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {arch_plot}')

compare_cols = ['Architecture', 'F1-Score', 'AUC', 'Accuracy', 'Class_0_Recall', 'Class_1_Recall', 'Train_Time_s', 'n_params']
sweep_compare = arch_sweep_df[compare_cols].copy()

print('\n=== ARCHITECTURE COMPARISON (same SMOTE ratio) ===')
display(sweep_compare.set_index('Architecture'))

best_arch = arch_sweep_df.loc[arch_sweep_df['F1-Score'].idxmax()]
print('\n=== BEST ARCHITECTURE (by test F1) ===')
for col in ['Architecture', 'hidden_layers', 'n_params', 'F1-Score', 'AUC', 'Accuracy', 'Train_Time_s', 'best_epoch']:
    print(f'  {col:<18}: {best_arch[col]}')

# Fastest among top-F1 configs (within 0.0001 F1 of best)
f1_tol = best_arch['F1-Score'] - 0.0001
fastest_good = arch_sweep_df[arch_sweep_df['F1-Score'] >= f1_tol].sort_values('Train_Time_s').iloc[0]
print('\n=== FASTEST NEAR-BEST (F1 within 0.0001 of top) ===')
for col in ['Architecture', 'hidden_layers', 'F1-Score', 'Train_Time_s', 'n_params']:
    print(f'  {col:<18}: {fastest_good[col]}')

BEST_ARCH_LAYERS = tuple(
    next(s['hidden_layers'] for s in FFNN_ARCHITECTURES if s['label'] == best_arch['Architecture'])
)

---
## Cell — Save Best FFNN Model


In [ ]:
# ── Retrain best FFNN and save artifacts ──────────────────────────────────
best_ratio = smote_results_df.loc[smote_results_df['F1-Score'].idxmax(), 'SMOTE_Ratio']
print(f'Retraining FFNN with best SMOTE ratio: {best_ratio}')

X_res, y_res, _ = resample_train_to_ratio(X_train, y_train, best_ratio)
selected_features, _ = select_features_xgb(X_res, y_res, X_test, 1.0)
X_tr = X_res[selected_features]
X_te = X_test[selected_features]

arch_layers = BEST_ARCH_LAYERS if 'BEST_ARCH_LAYERS' in globals() else (128, 64, 32)
print(f'Using architecture hidden_layers={arch_layers}')

best_model = create_ffnn(device=FFNN_DEVICE, hidden_layers=arch_layers)
fit_ffnn(best_model, X_tr, y_res)

y_pred = best_model.predict(X_te)
y_prob = best_model.predict_proba(X_te)[:, 1]

joblib.dump(best_model, MODEL_DIR / 'ffnn_binary_ddos.joblib')
joblib.dump(scaler, MODEL_DIR / 'scaler_ffnn.joblib')
with open(MODEL_DIR / 'selected_features_ffnn.json', 'w', encoding='utf-8') as f:
    json.dump(selected_features, f, indent=2)

report = {
    'model': MODEL_NAME,
    'hidden_layers': list(arch_layers),
    'best_smote_ratio': best_ratio,
    'metrics': build_smote_experiment_row(
        model_name=MODEL_NAME,
        y_test=y_test,
        y_pred=y_pred,
        y_prob=y_prob,
        ratio_label=best_ratio,
        resample_method='retrain',
        train_rows=len(X_res),
        train_time=0.0,
        n_features=len(selected_features),
    ),
    'device': str(FFNN_DEVICE),
    'selected_features': selected_features,
}
with open(MODEL_DIR / 'ffnn_training_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print(f'\nSaved model to {MODEL_DIR / "ffnn_binary_ddos.joblib"}')
print(f'Saved report to {MODEL_DIR / "ffnn_training_report.json"}')


---
## Download Results (Colab)

Run after experiments finish to save CSVs and plots to your computer.


In [ ]:
from google.colab import files
import shutil
from pathlib import Path

zip_base = '/content/ddos_outputs'
if Path(OUTPUT_DIR).exists() and any(Path(OUTPUT_DIR).iterdir()):
    shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
    print('Downloading outputs zip...')
    files.download(f'{zip_base}.zip')
else:
    print('No files in outputs yet — run experiment cells first.')
